# 🤖 Lekcja 4: Pierwszy model klasyfikacyjny, przeuczenie (*Overfitting*) i Macierz Pomyłek (*Confusion Matrix*)

W Lekcji 3 przygotowaliśmy zestaw **5 czystych, liczbowych cech** dla każdego pasażera: klasę biletu (`Pclass`), płeć (`Sex_Num`), wiek (`Age`), cenę biletu (`Fare`) oraz posiadanie kabiny (`Has_Cabin`).

Dziś zbudujemy nasz pierwszy model klasyfikacyjny — **Drzewo Decyzyjne (`DecisionTreeClassifier`)** — i poznamy trzy kluczowe pojęcia:

---

### 1. 🔒 Po co dzielimy dane na dwie części (`train_test_split`)? Analogia „Sejfu walidacyjnego”

Pamiętasz z Lekcji 1, że w pliku egzaminacyjnym `../data/test.csv` (418 pasażerów) **nie ma kolumny `Survived`** (prawdziwe odpowiedzi zna tylko serwis Kaggle)?  
Skąd więc mamy wiedzieć u siebie na komputerze, czy nasz model dobrze przewiduje los pasażerów, czy tylko zgaduje?

* Gdybyśmy nauczyli model na wszystkich 891 pasażerach z `train.csv`, a potem sprawdzili go na **tych samych 891 osobach**, byłoby to jak danie uczniowi na maturze dokładnie tych samych pytań, których odpowiedzi wykuł dzień wcześniej na pamięć!
* Dlatego za pomocą funkcji **`train_test_split`** przecinamy naszych 891 pasażerów z `train.csv` na dwie części:
  1. 📚 **Zbiór treningowy (`80%` = 712 pasażerów — `X_train`, `y_train`)**: z tej części model uczy się zależności.
  2. 🔒 **Zbiór walidacyjny, czyli nasz „Sejf” (`20%` = 179 pasażerów — `X_val`, `y_val`)**: tych 179 osób **zamykamy w sejfie**! Model **NIGDY nie widzi ich podczas nauki**. Dopiero gdy skończy się uczyć, wyjmujemy z sejfu cechy tych 179 pasażerów (`X_val`), prosimy model o wytypowanie, kto przeżył, a na końcu porównujemy jego odpowiedzi z prawdziwym kluczem odpowiedzi z sejfu (`y_val`)!

---

### 2. 🎯 Jak oceniamy model w klasyfikacji? Dokładność (*Accuracy*)
W klasyfikacji liczymy po prostu **odsetek trafnych odpowiedzi (*Accuracy*)**:

$$\text{Accuracy} = \frac{\text{Liczba poprawnie odgadniętych pasażerów}}{\text{Liczba wszystkich ocenianych pasażerów}} \times 100\%$$

Jeśli w naszym **sejfie walidacyjnym (`X_val`)** mamy **179 pasażerów**, a model poprawnie odgadł los **143 z nich**, to jego dokładność wynosi $\frac{143}{179} \approx \mathbf{79.9\%}$!

---

### 3. ⚠️ Pułapka kucia na pamięć (*Overfitting*) i hamulec bezpieczeństwa `max_depth`
Co się stanie, jeśli pozwolimy Drzewu Decyzyjnemu rosnąć bez żadnego ograniczenia?
* Drzewo będzie zadawać pytania tak długo (np. *„Czy to mężczyzna? Czy ma 27.5 roku? Czy zapłacił dokładnie 13.86 funta?”*), aż stworzy osobną gałąź dla każdego pojedynczego pasażera z książki treningowej!
* Na danych do nauki (`X_train`) osiągnie wtedy niemal **100% (`98.2%`)**, ale na nowych pasażerach wyjętych z sejfu (`X_val`) wypadnie słabo (**`77.1%`**), bo zamiast uczyć się ogólnych reguł, wykuło wyjątki na blachę!
* **Rozwiązanie (`max_depth=4`):** Ograniczamy głębokość drzewa do maksymalnie **4 pięter pytań** — zmuszając model, by skupił się tylko na najważniejszych regułach!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# 1. Wczytujemy dane z folderu ../data/ i przygotowujemy nasze 5 czystych cech z Lekcji 3
train = pd.read_csv('../data/train.csv')

train['Sex_Num'] = train['Sex'].map({'male': 0, 'female': 1})
train['Age'] = train['Age'].fillna(train['Age'].median())
train['Has_Cabin'] = train['Cabin'].notnull().astype(int)

cechy = ['Pclass', 'Sex_Num', 'Age', 'Fare', 'Has_Cabin']
X = train[cechy]
y = train['Survived']

# 2. Dzielimy dane: 80% do nauki (712 pasażerów), 20% do sejfu na egzamin (179 pasażerów)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Porównujemy Drzewo BEZ LIMITU głębokości oraz Drzewo OGRANICZONE (max_depth=4)
drzewo_bez_limitu = DecisionTreeClassifier(random_state=42)
drzewo_bez_limitu.fit(X_train, y_train)

drzewo_ograniczone = DecisionTreeClassifier(max_depth=4, random_state=42)
drzewo_ograniczone.fit(X_train, y_train)

print('--- 1. Drzewo BEZ LIMITU głębokości (kucie na pamięć / Overfitting) ---')
print(f"   Wynik na danych do nauki (X_train): {accuracy_score(y_train, drzewo_bez_limitu.predict(X_train))*100:.1f}%")
print(f"   Wynik na egzaminie w sejfie (X_val): {accuracy_score(y_val, drzewo_bez_limitu.predict(X_val))*100:.1f}%\n")

print('--- 2. Drzewo OGRANICZONE do 4 pięter pytań (max_depth=4) ---')
print(f"   Wynik na danych do nauki (X_train): {accuracy_score(y_train, drzewo_ograniczone.predict(X_train))*100:.1f}%")
print(f"   Wynik na egzaminie w sejfie (X_val): {accuracy_score(y_val, drzewo_ograniczone.predict(X_val))*100:.1f}%")

In [ ]:
# Zobaczmy na dwóch wykresach obok siebie:
# 1) Jak zmienia się dokładność na treningu i egzaminie przy zwiększaniu głębokości drzewa (max_depth od 1 do 15)
# 2) Macierz Pomyłek (Confusion Matrix) dla naszego najlepszego drzewa (max_depth=4)

glebokosci = list(range(1, 16))
wyniki_train = []
wyniki_val = []

for d in glebokosci:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    wyniki_train.append(accuracy_score(y_train, m.predict(X_train)) * 100)
    wyniki_val.append(accuracy_score(y_val, m.predict(X_val)) * 100)

pred_val = drzewo_ograniczone.predict(X_val)
macierz = confusion_matrix(y_val, pred_val)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

# WYKRES LEWY: Krzywa przeuczenia (Overfitting Curve)
ax1.plot(glebokosci, wyniki_train, '--o', color='royalblue', linewidth=2, label='Dokładność na nauce (X_train)')
ax1.plot(glebokosci, wyniki_val, '-s', color='mediumseagreen', linewidth=2.5, label='Dokładność na egzaminie w sejfie (X_val)')
ax1.axvline(4, color='red', linestyle=':', linewidth=2, label='Złoty środek (max_depth = 4 -> 79.9%)')
ax1.set_title('1. Efekt przeuczenia (Overfitting) a głębokość drzewa')
ax1.set_xlabel('Maksymalna głębokość drzewa (max_depth)')
ax1.set_ylabel('Dokładność — Accuracy (%)')
ax1.set_xticks(glebokosci)
ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.6)

# WYKRES PRAWY: Macierz Pomyłek (Confusion Matrix) dla 179 pasażerów z sejfu
ax2.imshow(macierz, cmap='Blues', alpha=0.75)
ax2.set_title('2. Macierz Pomyłek (Confusion Matrix) dla max_depth=4')
ax2.set_xlabel('Co przewidział nasz MODEL?')
ax2.set_ylabel('Jaka była PRAWDA historyczna?')
ax2.set_xticks([0, 1])
ax2.set_xticklabels(['Model: 0 (Zginął)', 'Model: 1 (Przeżył)'])
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Prawda: 0 (Zginął)', 'Prawda: 1 (Przeżył)'])

opisy_kratek = [
    ['TN = 92\n(Trafnie: Zginął)', 'FP = 13\n(Błąd: Fałszywy ocalały)'],
    ['FN = 23\n(Błąd: Przegapiony ocalały)', 'TP = 51\n(Trafnie: Przeżył)']
]
for i in range(2):
    for j in range(2):
        kolor_tekstu = 'darkgreen' if i == j else 'darkred'
        ax2.text(j, i, opisy_kratek[i][j], ha='center', va='center', fontsize=11, fontweight='bold', color=kolor_tekstu)

plt.tight_layout()
plt.show()

### 🔬 Jak czytać oba wykresy? (Krzywa przeuczenia i anatomia Macierzy Pomyłek)

#### 1. Lewy wykres — dowód na to, czym jest przeuczenie (*Overfitting*)
Spójrz na dwie linie na lewym wykresie:
* **Niebieska przerywana linia (nauka na `X_train`):** Im głębsze drzewo (5, 8, 12, 15 pięter), tym wyżej pnie się wynik na danych treningowych — aż do prawie 100%. Drzewo po prostu zapamiętuje konkretnych pasażerów.
* **Zielona ciągła linia (prawdziwy egzamin w sejfie `X_val`):** Rośnie tylko do **`max_depth = 3` lub `4` (`79.9%`)**, a potem przy głębszych drzewach **zaczyna spadać w dół**! Te rozwierające się „nożyce” między niebieską a zieloną linią to wizualny symbol przeuczenia (*Overfitting*).

---

#### 2. Prawy wykres — pełna anatomia Macierzy Pomyłek (*Confusion Matrix*)
Sam procent dokładności (`79.9%`) mówi nam, że model trafił **143 razy na 179 pasażerów**, ale nie mówi, **jakiego rodzaju błędy** popełnił na pozostałych 36 osobach. Do tego służy **Macierz Pomyłek (`confusion_matrix`)**, która dzieli wszystkich 179 pasażerów z sejfu na 4 kratki:

| Kratka w macierzy | Skrót angielski | Liczba osób w sejfie | Co dokładnie oznacza? |
| :--- | :---: | :---: | :--- |
| **Lewy górny róg** (Prawda `0`, Model `0`) | **TN** (*True Negative*) | **`92`** | ✅ **Trafna decyzja:** Pasażer niestety zginął (`0`), a model poprawnie przewidział `0`. |
| **Prawy dolny róg** (Prawda `1`, Model `1`) | **TP** (*True Positive*) | **`51`** | ✅ **Trafna decyzja:** Pasażer przeżył (`1`), a model poprawnie przewidział `1`. |
| **Prawy górny róg** (Prawda `0`, Model `1`) | **FP** (*False Positive*) | **`13`** | ❌ **Fałszywy alarm (Zbyt duży optymizm):** Pasażer zginął (`0`), ale model myślał, że przeżył (`1`) — np. młoda kobieta z 3. klasy, której nie udało się dotrzeć do szalupy. |
| **Lewy dolny róg** (Prawda `1`, Model `0`) | **FN** (*False Negative*) | **`23`** | ❌ **Przegapiony ratunek (Zbyt duży pesymizm):** Pasażer przeżył (`1`), ale model skazał go na porażkę (`0`) — np. dorosły mężczyzna, któremu mimo wszystko udało się wskoczyć do szalupy! |

Sprawdźmy matematykę z tej tabelki:
* Suma wszystkich 4 kratek: $92 + 13 + 23 + 51 = \mathbf{179}$ pasażerów w sejfie.
* Suma przekątnej sukcesu (`TN + TP`): $92 + 51 = \mathbf{143}$ trafne wytypowania ($\frac{143}{179} = \mathbf{79.9\%}$).

In [ ]:
# 🎯 ĆWICZENIE 1: Sprawdź, co robi najprostsze drzewo świata o głębokości 1 (max_depth=1)!
#
# Drzewo z parametrem `max_depth=1` może zadać tylko JEDNO pytanie (o płeć pasażera!).
# 1. Stwórz model `drzewo_1_pytanie = DecisionTreeClassifier(max_depth=1, random_state=42)`
# 2. Wytrenuj go na `(X_train, y_train)` metodą .fit()
# 3. Oblicz i wypisz jego dokładność na sejfie `X_val` za pomocą: accuracy_score(y_val, ...) * 100

# Miejsce na Twój kod:


In [ ]:
# 🎯 ĆWICZENIE 2: Detektyw błędów — kim byli pasażerowie, na których nasz model się pomylił?
#
# Poniżej przygotowałem tabelę `egzamin` dla 179 pasażerów z sejfu wraz z ich prawdziwym losem
# oraz decyzją naszego modelu `drzewo_ograniczone` (pred_val).

egzamin = train.loc[X_val.index, ['PassengerId', 'Name', 'Pclass', 'Sex', 'Age', 'Fare', 'Survived']].copy()
egzamin['Predykcja_Modelu'] = pred_val

# Wyfiltruj z tabeli `egzamin` wiersze, gdzie prawdziwy los różni się od decyzji modelu
# (użyj operatora `!=`, który w Pythonie oznacza „jest różne od”: egzamin['Survived'] != egzamin['Predykcja_Modelu'])
# i wyświetl pierwsze 10 takich osób (.head(10))!

# Miejsce na Twój kod:


---
### 📌 Podsumowanie lekcji w 3 punktach

1. **Dokładność (*Accuracy*) w klasyfikacji:** Mierzymy ją funkcją `accuracy_score(y_val, predykcje)`, która oblicza ułamek (lub procent) pasażerów, dla których model bezbłędnie odgadł `0` lub `1`.
2. **Walka z przeuczeniem (*Overfitting*) przez `max_depth`:** Nieograniczone drzewo decyzyjne uczy się danych treningowych na pamięć (`98.2%` na treningu, ale tylko `77.1%` w sejfie). Ograniczenie głębokości do `max_depth=4` podnosi wynik na egzaminie w sejfie do **`79.9%`**.
3. **Macierz Pomyłek (`confusion_matrix`):** Rozbija wynik na 4 kratki: trafnie wskazanych zmarłych (`TN = 92`), trafnie wskazanych ocalałych (`TP = 51`), fałszywie wytypowanych ocalałych (`FP = 13`) oraz przegapionych ocalałych (`FN = 23`).